# Testing AST and CFG

In [8]:
import ast


In [9]:
# Beispielcode wie in einer echten Datei
file_content = """
GLOBAL_A = 10
GLOBAL_B = 5

def unused():
    return 123

def helper(x):
    return x * GLOBAL_A

def another_helper(y):
    return helper(y) + GLOBAL_B

def f(n):
    return another_helper(n - 1)
"""

In [10]:
tree = ast.parse(file_content)
lines = file_content.splitlines(keepends=True)

In [11]:
class DependencyFinder(ast.NodeVisitor):
    def __init__(self, target_function_name):
        self.target = target_function_name
        self.inside = False
        self.called_functions = set()
        self.used_variables = set()

    def visit_FunctionDef(self, node):
        if node.name == self.target:
            self.inside = True
            self.generic_visit(node)
            self.inside = False
        elif self.inside:
            self.generic_visit(node)

    def visit_Call(self, node):
        if self.inside:
            if isinstance(node.func, ast.Name):
                self.called_functions.add(node.func.id)
        self.generic_visit(node)

    def visit_Name(self, node):
        if self.inside and isinstance(node.ctx, ast.Load):
            self.used_variables.add(node.id)
        self.generic_visit(node)


In [12]:
finder = DependencyFinder("f")
finder.visit(tree)

print("Aufgerufene Funktionen:", finder.called_functions)
print("Benutzte Variablen:", finder.used_variables)


Aufgerufene Funktionen: {'another_helper'}
Benutzte Variablen: {'n', 'another_helper'}


In [13]:
function_defs = {}
variable_defs = {}

for node in ast.walk(tree):
    # Funktionen sammeln
    if isinstance(node, ast.FunctionDef):
        if node.name in finder.called_functions or node.name == "f":
            function_defs[node.name] = node
    
    # Variablen sammeln
    if isinstance(node, ast.Assign):
        for t in node.targets:
            if isinstance(t, ast.Name) and t.id in finder.used_variables:
                variable_defs[t.id] = node


In [14]:
def extract(node):
    return "".join(lines[node.lineno - 1 : node.end_lineno])


In [15]:
print("\n=== Relevante Funktionen ===")
for name, node in function_defs.items():
    print(f"\n# Funktion: {name}\n{extract(node)}")

print("\n=== Relevante Variablen ===")
for name, node in variable_defs.items():
    print(f"\n# Variable: {name}\n{extract(node)}")



=== Relevante Funktionen ===

# Funktion: another_helper
def another_helper(y):
    return helper(y) + GLOBAL_B


# Funktion: f
def f(n):
    return another_helper(n - 1)


=== Relevante Variablen ===
